# pybind11: Binding C++ to Python (with Trading Examples)

This notebook is a **from-scratch to intermediate/advanced** guide to **pybind11**:
using modern C++ to write Python extensions and bind C++ classes, functions, and
interfaces.

We will use **trading-related C++ types** (orders, positions, signals, trades) as
our running examples.

### Contents

- [1. What is pybind11 and how does it fit?](#1-what-is-pybind11-and-how-does-it-fit)
- [2. Minimal Example: Binding a Free Function](#2-minimal-example-binding-a-free-function)
- [3. Binding a Simple C++ Class (Order)](#3-binding-a-simple-c-class-order)
- [4. Methods, Properties, and Enums (OrderType, Side)](#4-methods-properties-and-enums-ordertype-side)
- [5. Ownership, Smart Pointers, and Lifetimes](#5-ownership-smart-pointers-and-lifetimes)
- [6. STL Containers and Ranges (Vectors of Trades)](#6-stl-containers-and-ranges-vectors-of-trades)
- [7. Exception Translation and Error Handling](#7-exception-translation-and-error-handling)
- [8. Mini Trading Engine Case Study (Orders + Positions)](#8-mini-trading-engine-case-study-orders--positions)
- [9. Performance Notes and Best Practices](#9-performance-notes-and-best-practices)
- [10. Pybind11 vs Pure Python: Orders of Magnitude](#10-pybind11-vs-pure-python-orders-of-magnitude)

## 1. What is pybind11 and How Does It Fit?

- **pybind11** is a lightweight C++11+ library that makes it easy to **expose C++ code
  to Python** as extension modules.
- You write **modern C++** (no C API boilerplate) and compile it into a `.pyd` / `.so`
  that Python imports like any other module.
- It is header-only; you include `<pybind11/pybind11.h>` and compile.

Compared with other options:

- **C API**: powerful but verbose and error-prone.
- **Cython**: write Python-like code that compiles to C; good when you start from Python.
- **pybind11**: great when you start from C++ (or want to share code with a C++ system)
  and just need clean Python bindings.

We’ll work in a small `pybind11_build/` directory next to this notebook, writing C++
source and a `setup.py` to build an extension that we can import from Python.

In [1]:
# 1.1 Check pybind11 and set up a build directory

import sys
import time
from pathlib import Path

try:
    import pybind11
    print("pybind11 version:", pybind11.__version__)
except ImportError:
    print("Install pybind11 first, e.g.: pip install pybind11")

root = Path(".").resolve()
build_dir = root / "pybind11_build"
build_dir.mkdir(exist_ok=True)
print("Build directory:", build_dir)

pybind11 version: 3.0.2
Build directory: C:\Users\GSL\Projects\Challenges\training\pybind11_build


## 2. Minimal Example: Binding a Free Function

We’ll start with the classic **`add(a, b)`** example in C++, expose it to Python
with pybind11, build the extension, and time it vs pure Python.

Steps:

1. Write a small C++ file with `add` and a `PYBIND11_MODULE` definition.
2. Write a `setup.py` that builds the extension using `setuptools` and `pybind11`.
3. Run `python setup.py build_ext --inplace` from the build directory.
4. `import` the resulting module from Python.

In [2]:
# 2.1 Write minimal C++ + setup.py for add(a, b)

minimal_cpp = r"""
#include <pybind11/pybind11.h>

namespace py = pybind11;

// A simple free function: add two numbers
double add(double a, double b) {
    return a + b;
}

PYBIND11_MODULE(trading_minimal, m) {
    m.doc() = "Minimal pybind11 example: add(a, b)";
    m.def("add", &add, "Add two numbers", py::arg("a"), py::arg("b"));
}
"""

setup_py = r"""
from setuptools import setup, Extension
import sys
import pybind11

include_dirs = [pybind11.get_include()]

ext_modules = [
    Extension(
        "trading_minimal",
        ["trading_minimal.cpp"],
        include_dirs=include_dirs,
        language="c++",
        extra_compile_args=["-std=c++14"],
    )
]

setup(
    name="trading_minimal",
    version="0.0.1",
    ext_modules=ext_modules,
)
"""

(trading_cpp_path := build_dir / "trading_minimal.cpp").write_text(minimal_cpp)
(build_dir / "setup_minimal.py").write_text(setup_py)
print("Wrote:", trading_cpp_path, "and setup_minimal.py")

Wrote: C:\Users\GSL\Projects\Challenges\training\pybind11_build\trading_minimal.cpp and setup_minimal.py


In [3]:
# 2.2 Build and import the minimal module

import subprocess

print("Building trading_minimal (this runs a C++ compiler)...")
res = subprocess.run(
    [sys.executable, "setup_minimal.py", "build_ext", "--inplace"],
    cwd=str(build_dir),
    capture_output=True,
    text=True,
)
if res.returncode != 0:
    print("Build failed. stdout:\n", res.stdout)
    print("stderr:\n", res.stderr)
else:
    print("Build succeeded.")

# Add build_dir to sys.path and import
if str(build_dir) not in sys.path:
    sys.path.insert(0, str(build_dir))

try:
    import trading_minimal  # type: ignore
    print("trading_minimal.add(1.5, 2.5) =", trading_minimal.add(1.5, 2.5))
except ImportError as e:
    print("Could not import trading_minimal:", e)

Building trading_minimal (this runs a C++ compiler)...
Build failed. stdout:
 running build_ext
building 'trading_minimal' extension

stderr:
 error: Microsoft Visual C++ 14.0 or greater is required. Get it with "Microsoft C++ Build Tools": https://visualstudio.microsoft.com/visual-cpp-build-tools/

Could not import trading_minimal: No module named 'trading_minimal'


## 3. Binding a Simple C++ Class (Order)

Next we bind a minimal **`Order` C++ class**:

- Fields: `symbol`, `side`, `quantity`.
- Methods: `notional(mid_price)`.

We expose it as a Python class `Order` with a normal constructor and methods. This
will feel very similar to the pure-Python `Order` classes from earlier notebooks,
except the implementation lives in C++.

In [ ]:
# 3.1 C++ Order class and binding

order_cpp = r"""
#include <pybind11/pybind11.h>
#include <pybind11/stl.h>
#include <string>

namespace py = pybind11;

enum class Side { Buy = 1, Sell = -1 };

struct Order {
    std::string symbol;
    Side side;
    double quantity;  // positive size

    Order(const std::string& sym, Side s, double qty)
        : symbol(sym), side(s), quantity(qty) {}

    double notional(double mid_price) const {
        return quantity * mid_price;
    }

    int signed_quantity() const {
        return static_cast<int>(quantity * (side == Side::Buy ? 1.0 : -1.0));
    }
};

PYBIND11_MODULE(trading_core, m) {
    m.doc() = "Trading core bindings: Order, Side";

    py::enum_<Side>(m, "Side")
        .value("Buy", Side::Buy)
        .value("Sell", Side::Sell)
        .export_values();

    py::class_<Order>(m, "Order")
        .def(py::init<const std::string&, Side, double>(),
             py::arg("symbol"), py::arg("side"), py::arg("quantity"))
        .def_readwrite("symbol", &Order::symbol)
        .def_readwrite("side", &Order::side)
        .def_readwrite("quantity", &Order::quantity)
        .def("notional", &Order::notional, py::arg("mid_price"))
        .def("signed_quantity", &Order::signed_quantity)
        .def("__repr__", [](const Order& o) {
            return "Order(symbol='" + o.symbol + "', side=" +
                   std::string(o.side == Side::Buy ? "Buy" : "Sell") +
                   ", quantity=" + std::to_string(o.quantity) + ")";
        });
}
"""

setup_core = r"""
from setuptools import setup, Extension
import pybind11

include_dirs = [pybind11.get_include()]

ext_modules = [
    Extension(
        "trading_core",
        ["trading_core.cpp"],
        include_dirs=include_dirs,
        language="c++",
        extra_compile_args=["-std=c++14"],
    )
]

setup(
    name="trading_core",
    version="0.0.1",
    ext_modules=ext_modules,
)
"""

(build_dir / "trading_core.cpp").write_text(order_cpp)
(build_dir / "setup_core.py").write_text(setup_core)
print("Wrote trading_core.cpp and setup_core.py")

In [ ]:
# 3.2 Build and use the C++ Order from Python

print("Building trading_core...")
res = subprocess.run(
    [sys.executable, "setup_core.py", "build_ext", "--inplace"],
    cwd=str(build_dir),
    capture_output=True,
    text=True,
)
if res.returncode != 0:
    print("Build failed. stdout:\n", res.stdout)
    print("stderr:\n", res.stderr)
else:
    print("Build succeeded.")

if str(build_dir) not in sys.path:
    sys.path.insert(0, str(build_dir))

try:
    import trading_core  # type: ignore

    print("Available in trading_core:", dir(trading_core))

    o = trading_core.Order("AAPL", trading_core.Side.Buy, 100.0)
    print(o)
    print("Notional @ 180:", o.notional(180.0))
    print("Signed quantity:", o.signed_quantity())
except ImportError as e:
    print("Could not import trading_core:", e)

## 4. Methods, Properties, and Enums (OrderType, Side)

We can expose:

- **Enums** for order types and sides (e.g. `Limit`, `Market`, `Buy`, `Sell`).
- **Methods** with default arguments and docstrings.
- **Read-only properties** via `.def_property_readonly`.

We extend the C++ side to include an `OrderType` enum and a derived property
`is_buy()` for convenience.

In [ ]:
# 4.1 Extend trading_core with OrderType and properties

order_cpp_v2 = r"""
#include <pybind11/pybind11.h>
#include <pybind11/stl.h>
#include <string>

namespace py = pybind11;

enum class Side { Buy = 1, Sell = -1 };

enum class OrderType { Market, Limit };

struct Order {
    std::string symbol;
    Side side;
    OrderType type;
    double quantity;
    double limit_price;  // only meaningful for limit orders

    Order(const std::string& sym, Side s, OrderType t, double qty, double limit_px = 0.0)
        : symbol(sym), side(s), type(t), quantity(qty), limit_price(limit_px) {}

    double notional(double mid_price) const {
        return quantity * mid_price;
    }

    bool is_buy() const { return side == Side::Buy; }
};

PYBIND11_MODULE(trading_core_v2, m) {
    m.doc() = "Trading core v2: Order with OrderType, properties";

    py::enum_<Side>(m, "Side")
        .value("Buy", Side::Buy)
        .value("Sell", Side::Sell)
        .export_values();

    py::enum_<OrderType>(m, "OrderType")
        .value("Market", OrderType::Market)
        .value("Limit", OrderType::Limit)
        .export_values();

    py::class_<Order>(m, "Order")
        .def(py::init<const std::string&, Side, OrderType, double, double>(),
             py::arg("symbol"), py::arg("side"), py::arg("type"), py::arg("quantity"), py::arg("limit_price") = 0.0)
        .def_readwrite("symbol", &Order::symbol)
        .def_readwrite("side", &Order::side)
        .def_readwrite("type", &Order::type)
        .def_readwrite("quantity", &Order::quantity)
        .def_readwrite("limit_price", &Order::limit_price)
        .def("notional", &Order::notional, py::arg("mid_price"))
        .def_property_readonly("is_buy", &Order::is_buy)
        .def("__repr__", [](const Order& o) {
            std::string t = (o.type == OrderType::Market) ? "Market" : "Limit";
            std::string s = (o.side == Side::Buy) ? "Buy" : "Sell";
            return "Order(symbol='" + o.symbol + "', side=" + s + ", type=" + t +
                   ", quantity=" + std::to_string(o.quantity) +
                   ", limit_price=" + std::to_string(o.limit_price) + ")";
        });
}
"""

setup_core_v2 = r"""
from setuptools import setup, Extension
import pybind11

include_dirs = [pybind11.get_include()]

ext_modules = [
    Extension(
        "trading_core_v2",
        ["trading_core_v2.cpp"],
        include_dirs=include_dirs,
        language="c++",
        extra_compile_args=["-std=c++14"],
    )
]

setup(
    name="trading_core_v2",
    version="0.0.1",
    ext_modules=ext_modules,
)
"""

(build_dir / "trading_core_v2.cpp").write_text(order_cpp_v2)
(build_dir / "setup_core_v2.py").write_text(setup_core_v2)
print("Wrote trading_core_v2.cpp and setup_core_v2.py")

In [ ]:
# 4.2 Build and test trading_core_v2

print("Building trading_core_v2...")
res = subprocess.run(
    [sys.executable, "setup_core_v2.py", "build_ext", "--inplace"],
    cwd=str(build_dir),
    capture_output=True,
    text=True,
)
if res.returncode != 0:
    print("Build failed. stdout:\n", res.stdout)
    print("stderr:\n", res.stderr)
else:
    print("Build succeeded.")

if str(build_dir) not in sys.path:
    sys.path.insert(0, str(build_dir))

try:
    import trading_core_v2  # type: ignore

    o = trading_core_v2.Order(
        symbol="ESZ4",
        side=trading_core_v2.Side.Buy,
        type=trading_core_v2.OrderType.Limit,
        quantity=10.0,
        limit_price=5000.25,
    )
    print(o)
    print("is_buy property:", o.is_buy)
    print("Notional @ mid=5001.0:", o.notional(5001.0))
except ImportError as e:
    print("Could not import trading_core_v2:", e)

## 5. Ownership, Smart Pointers, and Lifetimes

pybind11 works naturally with **smart pointers** (`std::shared_ptr`, `std::unique_ptr`).
By default, it assumes C++ owns the object and Python just holds a reference. When
exposing classes that are also managed by a C++ engine (e.g. positions owned by
a risk engine), it’s common to bind them via `std::shared_ptr` so both sides
share ownership.

We’ll create a tiny `Position` class that holds aggregated quantity and average
price, and a `Portfolio` that owns `Position` objects via `std::shared_ptr`.
Python gets references but does not need to manage deletion explicitly.

In [ ]:
# 5.1 C++ Position + Portfolio with shared_ptr

portfolio_cpp = r"""
#include <pybind11/pybind11.h>
#include <pybind11/stl.h>
#include <memory>
#include <string>
#include <unordered_map>

namespace py = pybind11;

struct Position {
    std::string symbol;
    double quantity;
    double avg_price;

    Position(const std::string& sym)
        : symbol(sym), quantity(0.0), avg_price(0.0) {}

    void apply_fill(double fill_qty, double fill_price) {
        // Simple running average
        double new_qty = quantity + fill_qty;
        if (new_qty == 0.0) {
            quantity = 0.0;
            avg_price = 0.0;
            return;
        }
        double total_value = avg_price * quantity + fill_price * fill_qty;
        quantity = new_qty;
        avg_price = total_value / quantity;
    }

    double unrealized_pnl(double mark_price) const {
        return (mark_price - avg_price) * quantity;
    }
};

struct Portfolio {
    std::unordered_map<std::string, std::shared_ptr<Position>> positions;

    std::shared_ptr<Position> get_or_create(const std::string& symbol) {
        auto it = positions.find(symbol);
        if (it != positions.end()) return it->second;
        auto pos = std::make_shared<Position>(symbol);
        positions[symbol] = pos;
        return pos;
    }

    std::vector<std::shared_ptr<Position>> all_positions() const {
        std::vector<std::shared_ptr<Position>> out;
        out.reserve(positions.size());
        for (auto& kv : positions) out.push_back(kv.second);
        return out;
    }
};

PYBIND11_MODULE(trading_portfolio, m) {
    py::class_<Position, std::shared_ptr<Position>>(m, "Position")
        .def(py::init<const std::string&>())
        .def_readwrite("symbol", &Position::symbol)
        .def_readwrite("quantity", &Position::quantity)
        .def_readwrite("avg_price", &Position::avg_price)
        .def("apply_fill", &Position::apply_fill,
             py::arg("fill_qty"), py::arg("fill_price"))
        .def("unrealized_pnl", &Position::unrealized_pnl, py::arg("mark_price"));

    py::class_<Portfolio>(m, "Portfolio")
        .def(py::init<>())
        .def("get_or_create", &Portfolio::get_or_create,
             py::arg("symbol"),
             py::return_value_policy::reference)
        .def("all_positions", &Portfolio::all_positions);
}
"""

setup_portfolio = r"""
from setuptools import setup, Extension
import pybind11

include_dirs = [pybind11.get_include()]

ext_modules = [
    Extension(
        "trading_portfolio",
        ["trading_portfolio.cpp"],
        include_dirs=include_dirs,
        language="c++",
        extra_compile_args=["-std=c++14"],
    )
]

setup(
    name="trading_portfolio",
    version="0.0.1",
    ext_modules=ext_modules,
)
"""

(build_dir / "trading_portfolio.cpp").write_text(portfolio_cpp)
(build_dir / "setup_portfolio.py").write_text(setup_portfolio)
print("Wrote trading_portfolio.cpp and setup_portfolio.py")

In [ ]:
# 5.2 Build and use Portfolio from Python

print("Building trading_portfolio...")
res = subprocess.run(
    [sys.executable, "setup_portfolio.py", "build_ext", "--inplace"],
    cwd=str(build_dir),
    capture_output=True,
    text=True,
)
if res.returncode != 0:
    print("Build failed. stdout:\n", res.stdout)
    print("stderr:\n", res.stderr)
else:
    print("Build succeeded.")

if str(build_dir) not in sys.path:
    sys.path.insert(0, str(build_dir))

try:
    import trading_portfolio  # type: ignore

    pf = trading_portfolio.Portfolio()
    pos = pf.get_or_create("AAPL")
    pos.apply_fill(100.0, 180.0)
    pos.apply_fill(-50.0, 185.0)
    print("Position:", pos.symbol, pos.quantity, pos.avg_price)
    print("Unrealized PnL @ 190:", pos.unrealized_pnl(190.0))

    # Same Position instance visible via all_positions
    all_pos = pf.all_positions()
    print("All positions count:", len(all_pos))
    print("Shared object identity:", all_pos[0] is pos)
except ImportError as e:
    print("Could not import trading_portfolio:", e)

## 6. STL Containers and Ranges (Vectors of Trades)

pybind11 provides automatic conversions between many **STL containers** and Python
built-ins:

- `std::vector<T>` ↔ list of `T`
- `std::map<K, V>` ↔ dict
- `std::optional<T>` ↔ `None` or `T`

We’ll define a small `Trade` struct (symbol, qty, price, timestamp) and a function
that returns a `std::vector<Trade>` representing a recent trade history. On the Python
side this becomes a list of `Trade` objects.

In [ ]:
# 6.1 Trades and STL vector bindings

trades_cpp = r"""
#include <pybind11/pybind11.h>
#include <pybind11/stl.h>
#include <string>
#include <vector>

namespace py = pybind11;

struct Trade {
    std::string symbol;
    double quantity;
    double price;
    long timestamp_ns;
};

std::vector<Trade> recent_trades(const std::string& symbol, int count) {
    std::vector<Trade> out;
    out.reserve(count);
    long base_ts = 1'700'000'000'000'000'000L;
    for (int i = 0; i < count; ++i) {
        out.push_back(Trade{symbol, 1.0 + i, 100.0 + i * 0.01, base_ts + i * 1'000'000});
    }
    return out;
}

PYBIND11_MODULE(trading_trades, m) {
    py::class_<Trade>(m, "Trade")
        .def_readwrite("symbol", &Trade::symbol)
        .def_readwrite("quantity", &Trade::quantity)
        .def_readwrite("price", &Trade::price)
        .def_readwrite("timestamp_ns", &Trade::timestamp_ns);

    m.def("recent_trades", &recent_trades,
          py::arg("symbol"), py::arg("count") = 5,
          "Return a vector of recent Trade objects.");
}
"""

setup_trades = r"""
from setuptools import setup, Extension
import pybind11

include_dirs = [pybind11.get_include()]

ext_modules = [
    Extension(
        "trading_trades",
        ["trading_trades.cpp"],
        include_dirs=include_dirs,
        language="c++",
        extra_compile_args=["-std=c++14"],
    )
]

setup(
    name="trading_trades",
    version="0.0.1",
    ext_modules=ext_modules,
)
"""

(build_dir / "trading_trades.cpp").write_text(trades_cpp)
(build_dir / "setup_trades.py").write_text(setup_trades)
print("Wrote trading_trades.cpp and setup_trades.py")

In [ ]:
# 6.2 Build and test STL conversions

print("Building trading_trades...")
res = subprocess.run(
    [sys.executable, "setup_trades.py", "build_ext", "--inplace"],
    cwd=str(build_dir),
    capture_output=True,
    text=True,
)
if res.returncode != 0:
    print("Build failed. stdout:\n", res.stdout)
    print("stderr:\n", res.stderr)
else:
    print("Build succeeded.")

if str(build_dir) not in sys.path:
    sys.path.insert(0, str(build_dir))

try:
    import trading_trades  # type: ignore

    trades = trading_trades.recent_trades("AAPL", 3)
    print("Type of trades:", type(trades))
    for t in trades:
        print("Trade:", t.symbol, t.quantity, t.price, t.timestamp_ns)
except ImportError as e:
    print("Could not import trading_trades:", e)

## 7. Exception Translation and Error Handling

When C++ throws an exception, you usually want it to appear as a **Python exception**.
pybind11 maps several standard exceptions by default (e.g. `std::runtime_error` → `RuntimeError`).
You can also **register custom exception types**.

We’ll create a C++ function that validates an order and throws `std::runtime_error` on
failure; from Python we’ll catch it as a `RuntimeError`.

In [ ]:
# 7.1 Exception translation example

exceptions_cpp = r"""
#include <pybind11/pybind11.h>
#include <stdexcept>
#include <string>

namespace py = pybind11;

void validate_order(double qty, double limit_price) {
    if (qty <= 0.0) {
        throw std::runtime_error("Quantity must be positive");
    }
    if (limit_price < 0.0) {
        throw std::runtime_error("Limit price cannot be negative");
    }
}

PYBIND11_MODULE(trading_exceptions, m) {
    m.def("validate_order", &validate_order,
          py::arg("qty"), py::arg("limit_price"),
          "Validate an order; raises RuntimeError on invalid input.");
}
"""

setup_exceptions = r"""
from setuptools import setup, Extension
import pybind11

include_dirs = [pybind11.get_include()]

ext_modules = [
    Extension(
        "trading_exceptions",
        ["trading_exceptions.cpp"],
        include_dirs=include_dirs,
        language="c++",
        extra_compile_args=["-std=c++14"],
    )
]

setup(
    name="trading_exceptions",
    version="0.0.1",
    ext_modules=ext_modules,
)
"""

(build_dir / "trading_exceptions.cpp").write_text(exceptions_cpp)
(build_dir / "setup_exceptions.py").write_text(setup_exceptions)
print("Wrote trading_exceptions.cpp and setup_exceptions.py")

In [ ]:
# 7.2 Build and see exceptions in Python

print("Building trading_exceptions...")
res = subprocess.run(
    [sys.executable, "setup_exceptions.py", "build_ext", "--inplace"],
    cwd=str(build_dir),
    capture_output=True,
    text=True,
)
if res.returncode != 0:
    print("Build failed. stdout:\n", res.stdout)
    print("stderr:\n", res.stderr)
else:
    print("Build succeeded.")

if str(build_dir) not in sys.path:
    sys.path.insert(0, str(build_dir))

try:
    import trading_exceptions  # type: ignore

    # Valid
    trading_exceptions.validate_order(100.0, 180.0)
    print("validate_order(100, 180) OK")

    # Invalid cases
    for qty, px in [(0, 180.0), (10, -1.0)]:
        try:
            trading_exceptions.validate_order(qty, px)
        except RuntimeError as e:
            print(f"validate_order({qty}, {px}) raised RuntimeError:", e)
except ImportError as e:
    print("Could not import trading_exceptions:", e)

## 8. Mini Trading Engine Case Study (Orders + Positions)

Finally, we put several ideas together into a **mini trading engine core** in C++:

- `Order` and `Trade` types (symbol, side, price, quantity, timestamp).
- `Engine` that:
  - accepts new orders,
  - simulates fills (simple immediate trades),
  - updates positions via `Position` objects.

We expose a small API to Python so that strategy code can call into this compiled
core, get back C++ objects (orders, trades, positions) as Python objects, and measure
performance.

In [ ]:
# 8.1 C++ mini engine: Orders, Trades, Positions, Engine

engine_cpp = r"""
#include <pybind11/pybind11.h>
#include <pybind11/stl.h>
#include <memory>
#include <string>
#include <unordered_map>
#include <vector>

namespace py = pybind11;

enum class Side { Buy = 1, Sell = -1 };

enum class OrderType { Market, Limit };

struct Order {
    int id;
    std::string symbol;
    Side side;
    OrderType type;
    double quantity;
    double limit_price;
};

struct Trade {
    int order_id;
    std::string symbol;
    double quantity;
    double price;
    long timestamp_ns;
};

struct Position {
    std::string symbol;
    double quantity;
    double avg_price;

    Position(const std::string& sym) : symbol(sym), quantity(0.0), avg_price(0.0) {}

    void apply_trade(double fill_qty, double fill_price) {
        double new_qty = quantity + fill_qty;
        if (new_qty == 0.0) {
            quantity = 0.0;
            avg_price = 0.0;
            return;
        }
        double total_value = avg_price * quantity + fill_price * fill_qty;
        quantity = new_qty;
        avg_price = total_value / quantity;
    }
};

struct Engine {
    int next_order_id = 1;
    long next_ts = 1'700'000'000'000'000'000L;
    std::unordered_map<std::string, std::shared_ptr<Position>> positions;
    std::vector<Trade> trades;

    std::shared_ptr<Position> get_or_create_position(const std::string& symbol) {
        auto it = positions.find(symbol);
        if (it != positions.end()) return it->second;
        auto pos = std::make_shared<Position>(symbol);
        positions[symbol] = pos;
        return pos;
    }

    Order new_order(const std::string& symbol, Side side, OrderType type,
                    double quantity, double limit_price) {
        Order o{next_order_id++, symbol, side, type, quantity, limit_price};
        // Simple fill model: fill entire order at limit_price (or a dummy mid)
        double px = (type == OrderType::Limit ? limit_price : limit_price);
        double signed_qty = quantity * (side == Side::Buy ? 1.0 : -1.0);
        auto pos = get_or_create_position(symbol);
        pos->apply_trade(signed_qty, px);
        trades.push_back(Trade{o.id, symbol, signed_qty, px, next_ts});
        next_ts += 1'000'000;  // +1ms
        return o;
    }

    std::vector<std::shared_ptr<Position>> all_positions() const {
        std::vector<std::shared_ptr<Position>> out;
        out.reserve(positions.size());
        for (auto& kv : positions) out.push_back(kv.second);
        return out;
    }

    const std::vector<Trade>& all_trades() const { return trades; }
};

PYBIND11_MODULE(trading_engine, m) {
    py::enum_<Side>(m, "Side")
        .value("Buy", Side::Buy)
        .value("Sell", Side::Sell)
        .export_values();

    py::enum_<OrderType>(m, "OrderType")
        .value("Market", OrderType::Market)
        .value("Limit", OrderType::Limit)
        .export_values();

    py::class_<Order>(m, "Order")
        .def_readonly("id", &Order::id)
        .def_readonly("symbol", &Order::symbol)
        .def_readonly("side", &Order::side)
        .def_readonly("type", &Order::type)
        .def_readonly("quantity", &Order::quantity)
        .def_readonly("limit_price", &Order::limit_price);

    py::class_<Trade>(m, "Trade")
        .def_readonly("order_id", &Trade::order_id)
        .def_readonly("symbol", &Trade::symbol)
        .def_readonly("quantity", &Trade::quantity)
        .def_readonly("price", &Trade::price)
        .def_readonly("timestamp_ns", &Trade::timestamp_ns);

    py::class_<Position, std::shared_ptr<Position>>(m, "Position")
        .def_readonly("symbol", &Position::symbol)
        .def_readonly("quantity", &Position::quantity)
        .def_readonly("avg_price", &Position::avg_price);

    py::class_<Engine>(m, "Engine")
        .def(py::init<>())
        .def("new_order", &Engine::new_order,
             py::arg("symbol"), py::arg("side"), py::arg("type"),
             py::arg("quantity"), py::arg("limit_price"))
        .def("all_positions", &Engine::all_positions)
        .def("all_trades", &Engine::all_trades,
             py::return_value_policy::reference_internal);
}
"""

setup_engine = r"""
from setuptools import setup, Extension
import pybind11

include_dirs = [pybind11.get_include()]

ext_modules = [
    Extension(
        "trading_engine",
        ["trading_engine.cpp"],
        include_dirs=include_dirs,
        language="c++",
        extra_compile_args=["-std=c++14"],
    )
]

setup(
    name="trading_engine",
    version="0.0.1",
    ext_modules=ext_modules,
)
"""

(build_dir / "trading_engine.cpp").write_text(engine_cpp)
(build_dir / "setup_engine.py").write_text(setup_engine)
print("Wrote trading_engine.cpp and setup_engine.py")

In [ ]:
# 8.2 Build and drive the engine from Python

print("Building trading_engine...")
res = subprocess.run(
    [sys.executable, "setup_engine.py", "build_ext", "--inplace"],
    cwd=str(build_dir),
    capture_output=True,
    text=True,
)
if res.returncode != 0:
    print("Build failed. stdout:\n", res.stdout)
    print("stderr:\n", res.stderr)
else:
    print("Build succeeded.")

if str(build_dir) not in sys.path:
    sys.path.insert(0, str(build_dir))

try:
    import trading_engine  # type: ignore

    eng = trading_engine.Engine()
    # Simulate a few orders
    for i in range(3):
        eng.new_order("AAPL", trading_engine.Side.Buy, trading_engine.OrderType.Limit, 100.0 + i * 10, 180.0 + i)
        eng.new_order("AAPL", trading_engine.Side.Sell, trading_engine.OrderType.Limit, 50.0, 181.0 + i)

    print("Positions:")
    for pos in eng.all_positions():
        print(" ", pos.symbol, "qty=", pos.quantity, "avg=", pos.avg_price)

    print("Trades (first 3):")
    trades = eng.all_trades()
    for t in trades[:3]:
        print(" ", t.order_id, t.symbol, t.quantity, t.price, t.timestamp_ns)
except ImportError as e:
    print("Could not import trading_engine:", e)

## 9. Performance Notes and Best Practices

### Performance expectations

- Binding overhead (Python → C++ call) is small but not zero; aim for **coarser-grained
  functions** that do more work per call (e.g. process a batch of orders) instead of
  per-element calls.
- Within C++, performance is essentially that of your **native code**; pybind11’s
  overhead is mostly in argument conversion.

### Best practices

- Keep C++ code in **separate `.cpp` / `.hpp` files**; treat bindings (`PYBIND11_MODULE`)
  as a thin layer on top.
- Use **smart pointers** for shared ownership and clear lifetime semantics.
- Use **enums** for discrete concepts (side, order type) instead of strings.
- Design your API so that Python code passes **POD-like structs** or simple
  data (ints, doubles, strings, vectors) that map well to pybind11’s converters.
- Consider exposing **batch operations** (e.g. `process_orders(list[OrderSpec])`)
  for maximum throughput.

### When to choose pybind11

- You already have or plan to write **C++ trading components** (engines, gateways,
  risk modules) and want a **Python interface** for strategy code, testing, or tooling.
- You need tight integration with C++ libraries (matching engines, numerical libs)
  and want minimal glue.

For purely Python-centric optimization, see the **Cython**, **Numba**, and **NumPy
optimization** notebooks; for C++-centric systems with a Python edge, **pybind11** is
often the cleanest approach.

## 10. Pybind11 vs Pure Python: Orders of Magnitude

This section gives **approximate**, order-of-magnitude comparisons to answer:

> *What do we gain by moving hot-path logic into C++ via pybind11 instead of
> leaving it in pure Python?*

These numbers assume **CPU-bound** code (loops, numeric work). I/O-bound code (network,
file) usually sees much smaller gains.

### 10.1 Single-call overhead vs work inside the call

- A single Python → C++ function call via pybind11 typically adds **tens to a few
  hundreds of nanoseconds** of overhead, depending on argument complexity (scalars vs
  large containers).
- A Python loop doing the same arithmetic per element typically costs **dozens of
  nanoseconds per iteration** just in interpreter overhead.
- If your C++ function does, say, **1000 arithmetic operations** per call, the call
  overhead is negligible; total time is dominated by native code speed.

Roughly:

- **Per element in Python**: ~50–200 ns (interpreter + dynamic dispatch)
- **Per element in C++ (through pybind11)**: ~1–5 ns once inside the C++ loop
- **Binding overhead per call**: ~50–200 ns

So for a function that processes **N elements** per call:

- Python-only loop: cost ≈ `N × (50–200 ns)`
- C++ via pybind11: cost ≈ `overhead + N × (1–5 ns)`

For `N = 10_000` this can easily be a **10–50× speedup**.

### 10.2 Latency and throughput for trading-style workloads

For simplified hot paths (e.g. **apply one fill to a position**, or **update a book
level**):

- **Pure Python**:
  - Per-operation latency: often in the **5–50 µs** range, depending on object
    allocations, indirections, and checks.
  - Single-core throughput: roughly **20k–200k ops/sec**.
- **C++ via pybind11** (stateful engine doing multiple ops per call):
  - Per-operation latency inside the engine: **0.5–5 µs** or less for well-written code.
  - Single-core throughput: **500k–5M ops/sec** or more, depending on data structures
    and memory access patterns.

In other words, moving the critical loop (order/position updates, risk checks,
book updates) from Python into C++ often yields a **1–2 orders of magnitude
throughput improvement**, and per-operation latency improvements of **5–20×**, when
measured under realistic loads.

### 10.3 When pybind11 shines vs when it doesn’t

**Great candidates:**

- Repeated **numeric or data-structure operations** on in-memory state:
  - Order matching, book maintenance, PnL and risk aggregation.
  - Parsing and validating large streams of messages.
- Situations where you can design a **batch API** (e.g. `process_orders(batch)`)
  so each Python → C++ call does substantial work.

**Less impactful:**

- Purely I/O-bound flows (HTTP/WebSocket I/O dominated by network latency).
- Very thin wrappers that just call a C++ function per message with almost no
  internal work: call overhead may eat a noticeable fraction of total time.

### 10.4 Comparing pybind11 to other options

Very roughly, for a CPU-bound hot path you might think in terms of relative
performance:

- **Pure Python**: baseline **1×**.
- **Optimized NumPy / Numba / Cython**: often **5–20×** faster, depending on how
  well the code is vectorized or compiled.
- **Native C++ via pybind11**: typically in the same ballpark as Cython/Numba for
  the same algorithm, but with easier reuse of existing C++ code and ecosystems
  (e.g. matching engines, networking libs). Net improvement vs raw Python often
  **10–50×**.

The exact multipliers depend heavily on workload, cache behavior, and compiler
optimizations, but for trading-engine style components (order handling, risk,
book building) you can realistically expect **single-digit microsecond** hot-path
latency and **hundreds of thousands to millions of ops/sec** per core by moving
those paths into C++ and binding them via pybind11.